# Download KSSL soil chemical parameters for MIN3P

This notebook:

1. queries the USDA-NRCS Kellogg Soil Survey Laboratory (KSSL) Lab Data Mart with `py-soildb`
2. downloads all candidate pedons for each target soil series
3. extracts water pH and ammonium-acetate cation exchange capacity (CEC)
4. summarizes data completeness so candidate pedons and analytical methods can be inspected
5. assigns KSSL horizons to simplified **A**, **B**, and **C** MIN3P layers
6. exports horizon-level and aggregated chemical parameters to parquet files

The KSSL query is limited to the chemical properties table. The layer, pedon, and site fields needed to identify and aggregate the samples are added automatically by `fetch_ldm`.

## Important modeling choices

- pH is the 1:1 soil-to-water measurement (`ph_h2o`). It is an operational soil measurement and may not exactly equal the initial porewater pH required by MIN3P.
- CEC is measured by ammonium acetate buffered at pH 7 (`cec_nh4_ph_7`). This is preferable to mixing CEC values determined using different methods.
- A–B–C values are first thickness-weighted within each pedon, then averaged across pedons so pedons with many sampled subhorizons do not receive disproportionate weight.
- pH is averaged on the pH scale because the KSSL values are suspension measurements rather than aqueous hydrogen-ion concentrations.

For final simulations, inspect both the original horizon-level measurements and the aggregated layers.

In [ ]:
from __future__ import annotations

from pathlib import Path
import asyncio

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from byte_util.util import all_sites

from soildb import fetch_ldm

# Define a cache directory to cache KSSL queries
cache_dir = Path("./.cache")
if not cache_dir.exists():
    cache_dir.mkdir(parents=True, exist_ok=True)

# Define output directory for figures
plot_dir = Path("./plots/chem_plots")

## 1. Query KSSL by soil series name

`py-soildb` performs a two-stage Lab Data Mart query: it first identifies matching pedons and then retrieves horizon-level laboratory data. We use the same case-insensitive `samp_name` filter as the physical-parameter notebook, but request only the chemical properties table.

In [ ]:
redownload = False

async def fetch_series(series_name, overwrite_cached=False, cache_dir='./.cache'):
    """Fetch chemical data for all pedons matching a sampled taxon name

    Parameters:
    -----------
    series_name: str
        Name of series to fetch
    overwrite_cached: bool
        If true, overwrite cached file and query the series again (default: False)
    cache_dir: str or pathlib.Path
        Directory to use for cached data (default: ./.cache)

    Returns:
    --------
    df: pandas.DataFrame
        Series data from the Lab Data Mart chemical properties table"""

    cache_dir = Path(cache_dir)

    # Check if this data has already been downloaded
    cache_name = series_name.replace(' ', '_')
    cache_file = f"{cache_name}_kssl_chemical.parquet"
    cache_path = cache_dir / cache_file

    if cache_path.exists() and not overwrite_cached:
        return pd.read_parquet(cache_path)

    # Convert the site ID to the corresponding KSSL series name
    if series_name in ['HoustonBlack', 'Houston_Black']:
        search_name = 'Houston Black'
    else:
        search_name = series_name.replace('_', ' ')

    # Fetch Lab Data Mart chemical properties
    response = await fetch_ldm(
        x=search_name,
        what="samp_name",
        tables=["lab_chemical_properties"],
    )
    df = response.to_pandas().copy()

    # Drop reporting layers because they aren't true horizons
    report_mask = df['layer_type'] == 'reporting layer'
    if report_mask.sum() > 0:
        df.drop(df.index[report_mask], inplace=True)

    # Save to cache folder
    df.to_parquet(cache_path, index=False)

    return df

series_frames = []

for site in all_sites:
    print(f"Querying {site}...")

    frame = await fetch_series(
        site,
        cache_dir=cache_dir,
        overwrite_cached=redownload,
    )

    # Move series_name to first column
    frame = frame.copy()
    frame['series_name'] = site
    frame = frame[['series_name'] + [c for c in frame.columns if c != 'series_name']]

    print(f"\t{len(frame)} horizon records")
    series_frames.append(frame)

all_horizons = pd.concat(series_frames, ignore_index=True, sort=False)

print(f"Total downloaded rows: {len(all_horizons):,}")
print(f"Total columns: {all_horizons.shape[1]}")

## 2. Select pH and CEC columns

Several pH columns are available:
1. `ph_water_extract`: pH of the solution extracted from the sample with water. It is used as an indicator of the irrigated pH of soils with water
2. `ph_h2o`: pH measured in 1:1 soil to water suspension
3. `ph_cacl2`: pH measured in 1:2 soil to 0.01 M CaCl2; suppresses salt and seasonal effects
4. `ph_kcl`: pH measured in pH of a sample measured in 1 N potassium chloride at a 1:1 soil to solution ratio. If the pH in potassium chloride is less than pH in water, exchangeable aluminum is indicated.

For now, we'll use `cec_nh4_ph_7` for CEC (cation exchange capacity by ammonium acetate (NH4OAc)). It represents CEC when variable-charge sites are partially deprotonated and includes both permanent charge and a portion of the pH-dependent surface charge.

In [ ]:
ph_col = 'ph_h2o'
cec_col = 'cec_nh4_ph_7'

ph_method_col = f'{ph_col}_method'
cec_method_col = f'{cec_col}_method'

# Retain KSSL measurements useful for diagnosing soil-pH buffering
diagnostic_cols = [
    # Operational pH measurements
    'ph_h2o',
    'ph_cacl2',
    'ph_kcl',
    'ph_saturated_paste',

    # Exchange capacity, exchangeable bases, and acidity
    'cec_nh4_ph_7',
    'ca_nh4_ph_7',
    'mg_nh4_ph_7',
    'na_nh4_ph_7',
    'k_nh4_ph_7',
    'sum_of_nh4_ph_7_Ext_bases',
    'base_sat_nh4oac_ph_7',
    'acidity_bacl2_tea_ph_8_2',
    'sum_of_cations_cec_pH_8_2',
    'base_sat_sum_of_cations_ph_8_2',
    'aluminum_kcl_extractable',
    'ecec_base_plus_aluminum',
    'aluminum_saturation',
    'exchangeable_sodium',

    # Carbonate, gypsum, and organic matter
    'caco3_lt_2_mm',
    'corrected_gypsum_lt_2_mm',
    'total_carbon_ncs',
    'total_nitrogen_ncs',
    'organic_carbon_walkley_black',
    'estimated_organic_carbon',
    'carbon_to_nitrogen_ratio',

    # Saturation-extract solution chemistry
    'ca_satx',
    'mg_satx',
    'na_satx',
    'k_satx',
    'co3_satx',
    'hco3_satx',
    'cl_satx',
    'so4_satx',
    'no3_satx',
    'h20_satx',
    'electrical_conductivity_satx',
    'ec_predict_one_to_two',
    'total_estimated_salts_satx',
    'sodium_absorption_ratio',

    # Indicators of pedogenic and variable-charge materials
    'fe_dithionite_citrate_extract',
    'aluminum_dithionite_citrate',
    'fe_ammoniumoxalate_extractable',
    'aluminum_ammonium_oxalate',
    'silica_ammonium_oxalate',
]

n_total = len(all_horizons)
for col in diagnostic_cols:
    na_count = all_horizons[col].isna().sum()
    if n_total == na_count:
        print(f"Column `{col}` is completely NaN")

print(f"pH column: {ph_col}")
print(f"CEC column: {cec_col}")

## 3. Check horizon, pedon, and measurement coverage

In [ ]:
horizons = all_horizons.copy()

# Drop D and R horizons
for label in ['D', 'D1', 'D2', 'R']:
    mask = horizons['hzn_desgn'] == label
    horizons = horizons[~mask]

# Lump discontinuity soil horizons into major group
mask = horizons['hzn_desgn'].str.startswith('II')
horizons.loc[mask, 'hzn_desgn'] = horizons.loc[mask, 'hzn_desgn'].str[2:]
mask = horizons['hzn_desgn'].str.startswith('2')
horizons.loc[mask, 'hzn_desgn'] = horizons.loc[mask, 'hzn_desgn'].str[1:]
mask = horizons['hzn_master'].str.startswith('^')
horizons.loc[mask, 'hzn_master'] = horizons.loc[mask, 'hzn_master'].str[1:]

# Drop horizons without hzn_master or hzn_desgn
master_mask = horizons['hzn_master'] == ''
desgn_mask = horizons['hzn_desgn'] == ''
mask = np.logical_and(master_mask, desgn_mask)
horizons = horizons[~mask]

# If hzn_master is missing, fill it with first character of hzn_desgn
mask = horizons['hzn_master'] == ''
horizons.loc[mask, 'hzn_master'] = horizons.loc[mask, 'hzn_desgn'].str[:1]

# Drop "Base Sat" and "Mineralogy" rows
drop_layers = [1592, 30495, 36811, 163333, 15951714, 15951715]
for layer_key in drop_layers:
    mask = horizons['layer_key'] == layer_key
    horizons = horizons[~mask]

# Simplify to primary horizon (e.g., "BC" --> "B" and "CB" --> "C")
horizons['hzn_master'] = horizons['hzn_master'].str[0]
horizons['hzn_master'] = horizons['hzn_master'].replace({"O": "A"})
horizons['hzn_master'] = horizons['hzn_master'].replace({"E": "A"})

# Print all horizons that aren't A, B, C, O, E, or R
for i in horizons.index:
    hzn_label = horizons.loc[i, 'hzn_master']
    if hzn_label not in ['A', 'B', 'C']:
        print(
            f"Unusual horizon label: {hzn_label} "
            f"(series: {horizons.loc[i, 'series_name']}, "
            f"pedon: {horizons.loc[i, 'pedon_key']})"
        )

In [ ]:
# Print a summary of missing values in key columns
coverage_summary = (
    horizons
    .groupby('series_name', dropna=False)
    .agg(
        n_rows=('pedon_key', 'size'),
        n_pedons=('pedon_key', 'nunique'),
        measured_pH=(ph_col, 'count'),
        measured_CEC=(cec_col, 'count'),
        missing_pH=(ph_col, lambda x: x.isna().sum()),
        missing_CEC=(cec_col, lambda x: x.isna().sum()),
        missing_top_depth=('hzn_top', lambda x: x.isna().sum()),
        missing_bottom_depth=('hzn_bot', lambda x: x.isna().sum()),
    )
    .reset_index()
)

display(coverage_summary)

## 4. Assign KSSL horizons to A, B, and C MIN3P layers

In [ ]:
# Calculate sampled horizon thickness
horizons['thickness_cm'] = horizons['hzn_bot'] - horizons['hzn_top']

bad_thickness = horizons['thickness_cm'] <= 0
if bad_thickness.any():
    display(
        horizons.loc[
            bad_thickness,
            [
                'series_name',
                'pedon_key',
                'labsampnum',
                'hzn_master',
                'hzn_top',
                'hzn_bot',
            ],
        ]
    )
    raise ValueError('Found one or more horizons with non-positive thickness.')

## 5. Calculate additional metrics beyond pH and CEC

In [ ]:
# Ensure each column is numeric
for column in diagnostic_cols:
    horizons[column] = pd.to_numeric(horizons[column], errors='coerce',)

def add_ph_diagnostics(df):
    """Calculate quantities useful for diagnosing soil-pH buffering."""
    df = df.copy()

    cec = df['cec_nh4_ph_7'].replace(0, np.nan)
    base_cols = ['ca_nh4_ph_7', 'mg_nh4_ph_7', 'na_nh4_ph_7', 'k_nh4_ph_7']

    # Reconstruct the sum of exchangeable bases only when all four
    # exchangeable cations were measured
    df['sum_bases_calculated_meq_100g'] = df[base_cols].sum(axis=1, min_count=len(base_cols))

    df['sum_bases_meq_100g'] = df['sum_of_nh4_ph_7_Ext_bases'].combine_first(df['sum_bases_calculated_meq_100g'])

    # Base saturation and individual exchanger occupancy
    df['base_saturation_calculated_pct'] = 100 * df['sum_bases_meq_100g'] / cec
    df['base_saturation_pct'] = df['base_sat_nh4oac_ph_7'].combine_first(df['base_saturation_calculated_pct'])
    df['nonbase_exchange_pct'] = 100 - df['base_saturation_pct']

    base_total = df['sum_bases_calculated_meq_100g'].replace(0, np.nan)

    for cation in ['ca', 'mg', 'na', 'k']:
        column = f'{cation}_nh4_ph_7'
        df[f'{cation}_pct_of_bases'] = 100 * df[column] / base_total
        df[f'{cation}_exchange_pct_of_cec'] = (df[f'{cation}_pct_of_bases']
                                               * df['base_saturation_pct'] / 100)

    # Potential acidity and exchangeable-Al diagnostics
    df['base_saturation_ph_8_2_calculated_pct'] = (
        100 * df['sum_bases_meq_100g']
        / (df['sum_bases_meq_100g'] + df['acidity_bacl2_tea_ph_8_2']).replace(0, np.nan)
    )
    df['base_saturation_ph_8_2_pct'] = df['base_sat_sum_of_cations_ph_8_2'].combine_first(
        df['base_saturation_ph_8_2_calculated_pct'])

    df['aluminum_saturation_calculated_pct'] = (
        100 * df['aluminum_kcl_extractable']
        / (df['sum_bases_meq_100g'] + df['aluminum_kcl_extractable']).replace(0, np.nan))

    df['aluminum_saturation_pct'] = df['aluminum_saturation'].combine_first(
        df['aluminum_saturation_calculated_pct'])

    # Acid-neutralizing-capacity proxies
    # 1 wt% CaCO3 =
    # 10 g/kg / 100.0869 g/mol * 2 mol charge/mol
    df['carbonate_anc_molc_kg'] = df['caco3_lt_2_mm'] * 10 / 100.0869 * 2

    df['exchangeable_base_anc_molc_kg'] = df['sum_bases_meq_100g'] / 100

    df['potential_acidity_molc_kg'] = df['acidity_bacl2_tea_ph_8_2'] / 100

    # Saturation-extract alkalinity proxy
    # Both columns are reported in mmol charge/L
    df['satx_alkalinity_proxy_mmolc_L'] = df[['hco3_satx', 'co3_satx']].sum(axis=1, min_count=1)

    # Avoid treating carbonate carbon as organic carbon
    df['organic_carbon_pct'] = df['organic_carbon_walkley_black'].combine_first(df['estimated_organic_carbon'])

    # Calculate ratio of major ions in saturation extract
    cations = ['ca', 'mg', 'na', 'k']
    satx_cations = [f'{ion}_satx' for ion in cations]
    rat_cations = [f'{ion}_ratio' for ion in cations]
    df[rat_cations] = np.nan
    df['total_cations'] = np.nan
    ion_mask = df[satx_cations].notna().all(axis=1)
    df.loc[ion_mask, 'total_cations'] = df.loc[ion_mask, satx_cations].sum(axis=1)
    for satx_col, ratio_col in zip(satx_cations, rat_cations):
        df.loc[ion_mask, ratio_col] = df.loc[ion_mask, satx_col]/df.loc[ion_mask, 'total_cations']

    # Differences among operational pH measurements
    df['delta_ph_h2o_cacl2'] = df['ph_h2o'] - df['ph_cacl2']
    df['delta_ph_h2o_kcl'] = df['ph_h2o'] - df['ph_kcl']
    df['delta_ph_h2o_saturated_paste'] = df['ph_h2o'] - df['ph_saturated_paste']

    # Preserve the short names used elsewhere in this notebook
    df['pH'] = df['ph_h2o']
    df['CEC_meq_100g'] = df['cec_nh4_ph_7']

    return df

## 6. Check pH and CEC values

In [ ]:
# Run several checks to make sure the parameter values are reasonable
qaqc_checks = {
    ph_col: [
        ('outside [2, 12]', lambda s: ~s.between(2, 12)),
    ],
    cec_col: [
        ('less than or equal to 0', lambda s: s <= 0),
        ('unusually large (> 200)', lambda s: s > 200),
    ],
}

found_error = False
for parameter, checks in qaqc_checks.items():
    values = pd.to_numeric(
        horizons[parameter],
        errors='coerce',
    )

    for description, check in checks:
        mask = values.notna() & check(values)
        if mask.sum() > 0:
            found_error = True
            print(f"\n{'=' * 60}\n{parameter}\n{'=' * 60}")
            print(f"{description}: {mask.sum()} value(s)")

            display(
                horizons.loc[
                    mask,
                    [
                        'series_name',
                        'pedon_key',
                        'labsampnum',
                        'hzn_master',
                        parameter,
                    ],
                ]
            )

if not found_error:
    print('No odd parameter values were found.')

## 6. Aggregate pH and CEC to A, B, and C horizons

In [ ]:
def weighted_mean(values, weights):
    """Return a weighted mean after dropping missing values."""
    valid = values.notna() & weights.notna() & (weights > 0)

    if not valid.any():
        return np.nan

    return np.average(values[valid], weights=weights[valid])

# First calculate one thickness-weighted value for each pedon and layer
pedon_layer_rows = []

for (series, pedon, layer), df in horizons.groupby(['series_name', 'pedon_key', 'hzn_master']):
    if layer not in ['A', 'B', 'C']:
        continue

    row = {'series_name': series,
           'pedon_key': pedon,
           'layer': layer,}

    for parameter in diagnostic_cols:
        row[parameter] = weighted_mean(df[parameter], df['thickness_cm'])
        row[f'n_{parameter}_samples'] = df[parameter].notna().sum()

    pedon_layer_rows.append(row)

pedon_layers = pd.DataFrame(pedon_layer_rows)

# Calculate ratios and other diagnostic values from the
# thickness-weighted pedon values
pedon_layers = add_ph_diagnostics(pedon_layers)

id_cols = ['series_name', 'pedon_key', 'layer']
sample_count_cols = [c for c in pedon_layers if c.startswith('n_') and c.endswith('_samples')]
value_cols = [c for c in pedon_layers
              if c not in id_cols + sample_count_cols
              and pd.api.types.is_numeric_dtype(pedon_layers[c])]

# Define pandas aggregation methods
aggregation = {}
for parameter in value_cols:
    aggregation[parameter] = (parameter, 'mean',)
    aggregation[f'n_{parameter}_pedons'] = (parameter, 'count',)
for parameter in sample_count_cols:
    aggregation[parameter] = (parameter, 'sum',)
soil_chemical_params = pedon_layers.groupby(['series_name', 'layer']).agg(**aggregation)

# Add total pedon counts
pedon_counts = (pedon_layers
                .groupby(['series_name', 'layer'])['pedon_key'].nunique()
                .rename('n_pedons_total'))
soil_chemical_params = soil_chemical_params.join(pedon_counts)

# Add original KSSL horizon counts
horizon_counts = (
    horizons.loc[horizons['hzn_master'].isin(['A', 'B', 'C'])]
    .groupby(['series_name', 'hzn_master'])
    .size()
    .rename('n_horizon_records')
    .rename_axis(index=['series_name', 'layer'])
)
soil_chemical_params = soil_chemical_params.join(horizon_counts)

# Preserve analytical method codes represented in each
# site/master-horizon combination
def unique_strings(values):
    values = sorted(set(values.dropna().astype(str)))

    if not values:
        return None

    return ' | '.join(values)

# Then average across pedons so each pedon contributes one value per layer
index = pd.MultiIndex.from_product([all_sites, ['A', 'B', 'C']], names=['soil_series', 'layer'])

soil_chemical_params = soil_chemical_params.rename_axis(index=['soil_series', 'layer']).reindex(index)

# Put the most useful values first
first_cols = [
    'pH',
    'CEC_meq_100g',
    'caco3_lt_2_mm',
    'base_saturation_pct',
    'sum_bases_meq_100g',
    'base_saturation_ph_8_2_pct',
    'aluminum_saturation_pct',
    'carbonate_anc_molc_kg',
    'satx_alkalinity_proxy_mmolc_L',
    'electrical_conductivity_satx',
    'organic_carbon_pct',
]

soil_chemical_params = soil_chemical_params[first_cols + [c for c in soil_chemical_params if c not in first_cols]]

In [ ]:
# Print any soil/layer combinations without pH or CEC measurements
for parameter in ['pH', 'CEC_meq_100g', 'caco3_lt_2_mm', 'total_cations']:
    missing = soil_chemical_params[parameter].isna()

    if missing.any():
        print(f"Missing {parameter}:")
        for series, layer in soil_chemical_params.index[missing]:
            if (series == 'Yolo' and layer == 'B') or \
                (series == 'Pullman' and layer == 'C'):
                continue
            print(f"    {series} {layer}")

In [ ]:
# Based on missing values above, assume that Palouse C has similar pH and CEC to Palouse B
exchange_cols = ['ca_exchange_pct_of_cec', 'mg_exchange_pct_of_cec',
                 'na_exchange_pct_of_cec', 'k_exchange_pct_of_cec']
for col in ['pH', 'CEC_meq_100g'] + exchange_cols:
    soil_chemical_params.loc[('Palouse', 'C'), col] = soil_chemical_params.loc[('Palouse', 'B'), col]

# And assume that carbonate in Palouse A/C horizons is same as in Palouse B
soil_chemical_params.loc[('Palouse', ['A', 'C']), 'caco3_lt_2_mm'] = (
    soil_chemical_params.loc)[('Palouse', 'B'), 'caco3_lt_2_mm']

# And assume that porewater chemistry is homogeneous across Cecil/Flanagan/Kalamazoo/Palouse A/B/C
for site in ['Cecil', 'Flanagan', 'Kalamazoo', 'Palouse']:
    for cols in ['ca_ratio', 'mg_ratio', 'na_ratio', 'k_ratio', 'total_cations']:
        soil_chemical_params.loc[(site, ['B', 'C']), col] = soil_chemical_params.loc[(site, 'A'), col]

# Save to file
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
outfile = f'{s3_base_path}/input-data/processed-data/soil_chemical_parameters.parquet'
soil_chemical_params.to_parquet(outfile, index=True)

soil_chemical_params

## Box plots of pH and CEC

In [ ]:
from matplotlib.patches import Patch

horizon_names = ['A', 'B', 'C']
offsets = {'A': -0.28, 'B': 0.00, 'C': 0.28}
horizon_colors = {'A': '#7A5C47', 'B': '#B97A57', 'C': '#CDBFAE'}
width = 0.22

def add_horizon_boxplots(ax, parameter):
    """Add A, B, and C boxplots for each soil series."""
    for i, series in enumerate(all_sites):
        for horizon_name in horizon_names:
            values = pedon_layers.loc[(pedon_layers['series_name'] == series)
                                      & (pedon_layers['layer'] == horizon_name), parameter].dropna()

            if values.empty:
                continue

            boxplot = ax.boxplot([values], positions=[i + offsets[horizon_name]], widths=width,
                                 patch_artist=True, showmeans=True)

            for box in boxplot['boxes']:
                box.set_facecolor(horizon_colors[horizon_name])
                box.set_alpha(0.7)

    ax.set(xlim=(-0.6, len(all_sites) - 0.4), xticks=range(len(all_sites)), xticklabels=all_sites)
    ax.tick_params(axis='x', rotation=30)


# Plot pH for each soil series and horizon
fig, ax = plt.subplots(figsize=(10, 5), tight_layout=True)

add_horizon_boxplots(ax, 'pH')
ax.set(ylabel='pH', title=f'pH by soil series and horizon (using `{ph_col}`)')
legend_handles = [Patch(facecolor=horizon_colors[hn], alpha=0.7, label=f'{hn} horizon') for hn in horizon_names]
ax.legend(handles=legend_handles)

# Plot CEC for each soil series and horizon
fig, ax = plt.subplots(figsize=(10, 5), tight_layout=True)
add_horizon_boxplots(ax, 'CEC_meq_100g')
ax.set(ylabel='CEC (meq/100 g)', title=f'CEC by soil series and horizon (using `{cec_col}`)')
ax.legend(handles=legend_handles)

## Compare pH measurements by site

In [ ]:
all_ph_cols = {'ph_h2o': 'pH in H$_2$O', 'ph_cacl2': 'pH in CaCl$_2$', 'ph_kcl': 'pH in KCl',}

fig, ax = plt.subplots(figsize=(10, 5), tight_layout=True)

for i, series in enumerate(all_sites):
    for j, ph_column in enumerate(all_ph_cols.keys()):
        values = horizons.loc[(horizons['series_name'] == series), ph_column].dropna()

        if values.empty:
            continue

        boxplot = ax.boxplot([values], positions=[i + list(offsets.values())[j]], widths=width,
                             patch_artist=True, showmeans=True)

        for box in boxplot['boxes']:
            box.set_facecolor(list(horizon_colors.values())[j])
            box.set_alpha(0.7)

ax.set(xlim=(-0.6, len(all_sites) - 0.4), xticks=range(len(all_sites)), xticklabels=all_sites, ylabel='pH measurement')
ax.tick_params(axis='x', rotation=30)

legend_handles = [Patch(facecolor=list(horizon_colors.values())[i], alpha=0.7, label=label)
                  for i, (cn, label) in enumerate(all_ph_cols.items())]
ax.legend(handles=legend_handles)

## Additional plots of exchangeable cations, carbonate content, etc.

In [ ]:
horizon_calcs = add_ph_diagnostics(horizons)
box_legend = [Patch(facecolor=horizon_colors[layer], alpha=0.7, label=f'{layer} horizon',) for layer in horizon_names]

def add_stripplot(ax, parameter, *, jitter=0.07, marker_size=24, seed=42):
    """Plot individual pedon values by site and A/B/C horizon as a stripplot."""
    rng = np.random.default_rng(seed)

    for i, series in enumerate(all_sites):
        for layer in horizon_names:
            mask = ((horizon_calcs['series_name']  == series) & (horizon_calcs['hzn_master'] == layer))
            values = horizon_calcs.loc[mask, parameter].dropna()

            if values.empty:
                continue
            center = i + offsets[layer]

            # Add deterministic horizontal jitter around each horizon position
            x = center + rng.uniform(-jitter, jitter, size=len(values))
            ax.scatter(x, values, s=marker_size, facecolor=horizon_colors[layer],
                       edgecolor='black', linewidth=0.4, alpha=0.75, zorder=2)

    ax.set(xlim=(-0.6, len(all_sites) - 0.4), xticks=range(len(all_sites)), xticklabels=all_sites)
    ax.tick_params(axis='x', rotation=30)

## Base saturation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), tight_layout=True)

add_stripplot(ax, 'base_saturation_pct')

ax.set(ylim=(0, 110), title='NH$_4$OAc base saturation')
ax.legend(handles=box_legend)

In [ ]:
comp_cols = {
    'Ca': 'ca_exchange_pct_of_cec',
    'Mg': 'mg_exchange_pct_of_cec',
    'Na': 'na_exchange_pct_of_cec',
    'K': 'k_exchange_pct_of_cec',
    'Other / nonbase': 'nonbase_exchange_pct',
}

complete_comp = horizon_calcs[list(comp_cols.values())].notna().all(axis=1)
comp_data = horizon_calcs.loc[complete_comp].copy()
exchange_comp = comp_data.groupby(['series_name', 'hzn_master'])[list(comp_cols.values())].mean()

n_complete = comp_data.groupby(['series_name', 'hzn_master']).size()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True, tight_layout=True)

for ax, layer in zip(axes, horizon_names):
    layer_index = pd.MultiIndex.from_product([all_sites, [layer]],
                                             names=['series_name', 'hzn_master'])

    layer_data = exchange_comp.reindex(layer_index).droplevel('hzn_master')
    layer_n = n_complete.reindex(layer_index).droplevel('hzn_master').fillna(0)
    bottom = np.zeros(len(all_sites))

    for label, column in comp_cols.items():
        values = layer_data[column].fillna(0).to_numpy()

        ax.bar(range(len(all_sites)), values, bottom=bottom, label=label,)
        bottom += values

    for i, n in enumerate(layer_n):
        ax.text(i, 102, f'n={int(n)}', ha='center', va='bottom', rotation=90, fontsize=7)

    ax.set(title=f'{layer} horizon', xticks=range(len(all_sites)),
           xticklabels=all_sites, ylim=(0, 112))
    ax.tick_params(axis='x', rotation=45)

axes[0].set_ylabel('Exchange-complex composition (%)')
axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')

fig.suptitle('Estimated exchangeable-cation composition by soil series and horizon')

In [ ]:
fig, ax = plt.subplots(5, figsize=(11, 18), sharex=True, tight_layout=True)
cols_to_plot = ['ca_ratio', 'mg_ratio', 'na_ratio', 'k_ratio', 'total_cations']
for i, col in enumerate(cols_to_plot):
    add_stripplot(ax[i], col)
    ion = col.split('_')[0]
    ion_str = f'{ion[0].upper()}{ion[1:]}'
    if i < 4:
        ax[i].set(ylim=(0, 1), ylabel=f'{ion_str}/(Ca + Mg + Na + K)', title=ion_str)

ax[4].set(ylabel='Total Ca + Mg + Na + K', title='Ca + Mg + Na + K')
ax[0].legend(handles=box_legend)
fig.suptitle('Major ion chemistry in saturation extract')

In [ ]:
plot_data = horizon_calcs.dropna(subset=['pH', 'base_saturation_pct', 'CEC_meq_100g',]).copy()
max_cec = plot_data['CEC_meq_100g'].max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True, tight_layout=True)

for ax, layer in zip(axes, horizon_names):
    layer_data = plot_data[plot_data['hzn_master'] == layer]

    for series in all_sites:
        data = layer_data[layer_data['series_name'] == series]

        if data.empty:
            continue

        point_sizes = (25 + 175* data['CEC_meq_100g'] / max_cec)
        ax.scatter(data['pH'], data['base_saturation_pct'], s=point_sizes, alpha=0.65, label=series)
    ax.set(title=f'{layer} horizon', xlabel='pH in H$_2$O', xlim=(4, 10))
axes[0].set_ylabel('Base saturation (%)')

# Combine legend entries across panels
legend_entries = {}
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_entries.setdefault(label, handle)
axes[0].legend(legend_entries.values(), legend_entries.keys())

fig.suptitle('pH versus base saturation; marker size indicates CEC')

In [ ]:
plot_data = horizon_calcs.dropna(subset=['pH', 'base_saturation_pct', 'CEC_meq_100g',]).copy()
max_cec = plot_data['CEC_meq_100g'].max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True, tight_layout=True)

for ax, layer in zip(axes, horizon_names):
    layer_data = plot_data[plot_data['hzn_master'] == layer]

    for series in all_sites:
        data = layer_data[layer_data['series_name'] == series]

        if data.empty:
            continue

        point_sizes = (25 + 175* data['CEC_meq_100g'] / max_cec)
        ax.scatter(data['pH'], data['aluminum_kcl_extractable'], s=point_sizes, alpha=0.65, label=series)
    ax.set(title=f'{layer} horizon', xlabel='pH in H$_2$O', xlim=(4, 10))
axes[0].set_ylabel('KCl-extractable Al (meq/100 g)')

# Combine legend entries across panels
legend_entries = {}
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_entries.setdefault(label, handle)
axes[0].legend(legend_entries.values(), legend_entries.keys())

fig.suptitle('KCl-extractable Al by pH; marker size indicates CEC')

## Carbonate content

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), tight_layout=True)

add_stripplot(ax, 'caco3_lt_2_mm')
ax.set(ylabel='CaCO$_3$ equivalent (<2 mm, wt. %)', title='Carbonate content')

ax.legend(handles=box_legend)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), tight_layout=True,)

add_stripplot(axes[0], 'acidity_bacl2_tea_ph_8_2')
add_stripplot(axes[1], 'aluminum_saturation_pct')

axes[0].set(ylabel='Potential acidity (cmol$_c$ kg$^{-1}$)', title='BaCl$_2$-TEA acidity to pH 8.2')
axes[1].set(ylabel='Aluminum saturation (%)', title='KCl-extractable aluminum saturation')
axes[1].set_ylim(0, 110)

axes[1].legend(handles=box_legend)

In [ ]:
fig, axes = plt.subplots(2, figsize=(12, 10), tight_layout=True)

add_stripplot(axes[0], 'satx_alkalinity_proxy_mmolc_L')
add_stripplot(axes[1], 'electrical_conductivity_satx')

axes[0].set(ylabel='HCO$_3^-$ + CO$_3^{2-}$ (mmol$_c$ L$^{-1}$)',
            title='Saturation-extract alkalinity proxy',)
axes[1].set(ylabel='Electrical conductivity (dS m$^{-1}$)',
            title='Saturation-extract electrical conductivity',)
axes[1].legend(handles=box_legend)